In [13]:
#%%capture
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
#!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

In [14]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [ ]:
!pip install mistralai==2.4.6

In [16]:
import pandas as pd
import os
import pickle
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [17]:
# 1. Read Stream from Kafka
raw_stream = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "192.168.1.153:8097") \
  .option("subscribe", "input") \
  .option("startingOffsets", "latest") \
  .load()

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [18]:
processed_data = raw_stream.select(
    col("key").cast(StringType()).alias("filename"),
    col("value").alias("png_data")
)

In [19]:
#import requests
from pyspark.sql import Row
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json
import base64
import os
import time
# ---------------------

# Initialize Mistral client with API key
from mistralai import Mistral
api_key = 'Zs7BIWkcpcsPfI2fgRYkWdbTUSb8FJR4'
client = Mistral(api_key=api_key)

In [20]:
def call_mistral_ocr(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    #return filename, image_response, base64_data_url
    # Convert response to JSON
    #response_dict = json.loads(image_response.model_dump_json())
    #ocr_result = json.dumps(response_dict, indent=4)
    #print(json_string)

    #print(image_response)

    #usage = image_response.usage

    #response_dict = json.loads(image_response.choices[0].message.content)
    
    #return filename, json.dumps(response_dict, indent=4), usage.prompt_tokens, usage.completion_tokens, usage.total_tokens

    #return filename, ocr_result #, usage.prompt_tokens, usage.completion_tokens, usage.total_tokens

    
    # This shows the entire JSON structure returned by the API
    print(json.dumps(image_response.model_dump(), indent=2))

    response_dict = json.loads(image_response.model_dump_json())
    #print(response_dict["pages"][0]["markdown"])
    json_string = json.dumps(response_dict["pages"][0]["markdown"], indent=4)
    return filename, json_string

In [21]:
!pip install jiwer

In [22]:
from jiwer import cer, wer

def evaluate_ocr_accuracy(ground_truth: str, ocr_text: str):
    """Calculate CER and WER between ground truth and OCR output."""
    character_error_rate = cer(ground_truth, ocr_text)
    word_error_rate = wer(ground_truth, ocr_text)

    print(f"CER: {character_error_rate:.4f} ({character_error_rate*100:.2f}%)")
    print(f"WER: {word_error_rate:.4f} ({word_error_rate*100:.2f}%)")
    return character_error_rate, word_error_rate


In [23]:
def process_batch(df, batch_id):
    """
    Function executed for every micro-batch of the Structured Stream.
    Collects the rows and processes them with the OCR function.
    """

    start_time_full = time.time()
    
    # ⚠️ WARNING: .collect() brings ALL data to the driver program. 
    # Use for testing/low-volume only. For high-volume, consider a custom Kafka Connect Sink.
    rows = df.collect()
    
    results = []
    c = 1
    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0
    
    for row in rows:
        print("c = ",c)
        #filename, ocr_text, input_tok, output_tok, total_tok = call_mistral_ocr(row)
        filename, ocr_text = call_mistral_ocr(row)
        print(f"filename = {filename}, ocr text = {ocr_text}")
        
        #filename, ocr_text, input_tok, output_tok, total_tok = call_mistral_8b_latest(row)
        #filename, orc_text = structured_ocr(row)
        #filename, structured_response = structured_ocr(row) # Process image and extract data
        #response_dict = json.loads(structured_response.model_dump_json())
        #print(json.dumps(response_dict, indent=4))

        #total_input_tokens += input_tok
        #total_output_tokens += output_tok
        #total_tokens += total_tok
        
        if filename is None:
            filename = f"image_{batch_id}"

        # 1. Convert string back to a dictionary
        #data_dict = json.loads(ocr_text)

        # 2. Add new data
        # Method A: Manual assignment
        #data_dict['total_input_tokens'] = total_input_tokens
        #data_dict['total_output_tokens'] = total_output_tokens
        #data_dict['total_tokens'] = total_tokens

        ground_truth_path = "/home/jovyan/g1.txt"

        if ground_truth_path and os.path.exists(ground_truth_path):

            with open(ground_truth_path, 'r', encoding='utf-8') as f:
                ground_truth = json.load(f) if ground_truth_path.endswith('.json') else f.read().strip()

            print(f"\n{'='*50}")
            print("ACCURACY EVALUATION")
            print(f"{'='*50}")
            char_err, word_err = evaluate_ocr_accuracy(ground_truth, ocr_text) #json.dumps(ocr_text, ensure_ascii=False))
            print(f"cer = {char_err}, and wer = {word_err}")
            #data_dict['char_err'] = char_err
            #data_dict['word_err'] = word_err

        # 3. Convert back to JSON string
        #result_str = json.dumps(data_dict, indent=4)

        #print(result_str)

        #results.append((filename, result_str))        
        #print(f"OCR Result for {filename}: {result_str}") # Log first 80 chars
        c = c+1

    schema = StructType([
        StructField("filename", StringType(), False),
        StructField("ocr_text", StringType(), False) 
    ])

    if results:
        
        results_df = spark.createDataFrame(results, schema)

        # 3. Write results to JSON files (The Solution)
        # Use the batch_id to create a unique subdirectory for the JSON files
        batch_output_path = os.path.join("/home/jovyan/json", f"batch_{batch_id}")

        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)
        
        print(f"Successfully wrote OCR results for Batch {batch_id} to: {batch_output_path}")
        
        end_time_full = time.time()
        
        total_duration = end_time_full - start_time_full
        
        print(f"   > Total Function Duration: {total_duration:.4f} seconds")
    

In [24]:
query = processed_data.writeStream \
    .foreachBatch(process_batch) \
    .start() 

query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = a9eee965-5c22-4d62-905b-fce81e962dd5, runId = f9d07442-e052-4678-9f10-a991237763fa] terminated with exception: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: describeTopics

In [ ]:
for q in spark.streams.active:
    q.stop()

In [ ]:
#Mistral tutorial https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/ocr/structured_ocr.ipynb#scrollTo=po7Cukllt8za